# Exam Ready Master Notebook (02504)

Fast entrypoint for computational questions. Use `Cmd+F` with `TEMPLATE:` headings.

In [ ]:
import numpy as np
import cv2

from exam_toolkit import *
from exam_templates import exam2023_templates as t23
from exam_templates import exam2024_templates as t24
from exam_templates import predicted_question_templates as tp

## Quick Index
- TEMPLATE: projection
- TEMPLATE: resize-K
- TEMPLATE: distortion
- TEMPLATE: homography
- TEMPLATE: epipolar-distance
- TEMPLATE: triangulation
- TEMPLATE: harris
- TEMPLATE: calibration
- TEMPLATE: ransac-formulas
- TEMPLATE: structured-light

## TEMPLATE: projection

In [ ]:
K = camera_intrinsic(1400, (750, 520))
rvec = np.array([0.2, 0.2, -0.1])
t = np.array([[-0.08], [0.01], [0.03]])
Q = np.array([[-0.38], [0.10], [1.32]])
p = t24.q1_projection(K, rvec, t, Q)
p

## TEMPLATE: resize-K

In [ ]:
K0 = np.array([[1000.0, 0.0, 400.0], [0.0, 1000.0, 350.0], [0.0, 0.0, 1.0]])
K_half = resize_intrinsics(K0, 0.5, 0.5)
K_half

## TEMPLATE: distortion

In [ ]:
K = np.array([[300, 0, 840], [0, 300, 620], [0, 0, 1]], float)
dist = [-0.2, 0.01, -0.03]
pixel = np.array([[400], [500]])
qd = t24.q4_distortion_mapping(K, dist, pixel)
qd

## TEMPLATE: homography

In [ ]:
p1 = np.array([[145.349587, -0.112915131, 1.91640565, -0.608129962],
               [105.60382, 0.0562792554, 1.79040110, -0.232182177]])
p2 = np.array([[1.3753556, -1.77072961, 2.94511795, 0.04032374],
               [0.30936653, 0.37172814, 1.44007577, -0.03173825]])
H = t24.q2_homography_from_4_points(p1, p2)
H / H[0,0]

## TEMPLATE: epipolar-distance

In [ ]:
K = np.array([[300, 0, 840], [0, 300, 620.0], [0, 0, 1]], float)
R1 = rodrigues_to_matrix(np.array([-2.3, -0.7, 1.0]))
t1 = np.array([[0.0], [-1.0], [4.0]])
R2 = rodrigues_to_matrix(np.array([-0.6, 0.5, -0.9]))
t2 = np.array([[0.0], [0.0], [9.0]])
p1 = np.array([[853.0], [656.0]])
p2 = np.array([[814.0], [655.0]])
d = t24.q15_epipolar_distance(K, R1, t1, R2, t2, p1, p2)
d

## TEMPLATE: triangulation

In [ ]:
R3 = rodrigues_to_matrix(np.array([-0.1, 0.9, -1.2]))
t3 = np.array([[-1.0], [-6.0], [28.0]])
P1 = projection_matrix(K, R1, t1)
P2 = projection_matrix(K, R2, t2)
P3 = projection_matrix(K, R3, t3)
p3 = np.array([[798.0], [535.0]])
Q_lin = triangulate_linear([p1, p2, p3], [P1, P2, P3])
Q_nonlin = triangulate_nonlinear([p1, p2, p3], [P1, P2, P3])
Q_lin, Q_nonlin

## TEMPLATE: harris

In [ ]:
h = np.load('notebooks/materials_2024/harris.npy', allow_pickle=True).item()
r, corners4 = t24.q6_harris_from_tensor(h['g*(I_x^2)'], h['g*(I_y^2)'], h['g*(I_x I_y)'], k=0.06, tau=5, connectivity=4)
_, corners8 = t24.q6_harris_from_tensor(h['g*(I_x^2)'], h['g*(I_y^2)'], h['g*(I_x I_y)'], k=0.06, tau=5, connectivity=8)
corners4.shape, corners8.shape

## TEMPLATE: ransac-formulas

In [ ]:
N = ransac_iterations_required(465, 1177, sample_size=4, confidence=0.90)
tau2 = squared_reprojection_threshold(1.4, chi_square_value=3.84)
N, tau2

## TEMPLATE: structured-light

In [ ]:
primary = np.array([12, 9, 10, 13, 18, 25, 33, 40, 46, 49, 48, 45, 39, 31, 23, 17])
secondary = np.array([15, 29, 43, 49, 43, 29, 15, 10])
theta = structured_light_unwrap_phase(primary, secondary, n1=40)
theta

## TEMPLATE: predicted-variations

In [ ]:
# Example predicted variation: convert E->F when K1,K2 known
E = np.array([[0, -1, 2], [1, 0, -3], [-2, 3, 0]], float)
K1 = camera_intrinsic(900, (1070, 610))
K2 = K1.copy()
F = tp.template_convert_E_and_F(E, K1, K2, input_type='E')
F